In [1]:
from pyspark.sql import SparkSession
import pandas as pd

In [2]:
#Starting the spark session
spark = SparkSession.builder.master("local[*]").appName("Great Expectations with Pandas DataFrame").getOrCreate()  

22/07/07 15:02:47 WARN Utils: Your hostname, DESKTOP-AOB9SOH resolves to a loopback address: 127.0.1.1; using 192.168.20.168 instead (on interface eth0)
22/07/07 15:02:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
22/07/07 15:02:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
22/07/07 15:02:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Loading Data: (with some exploration) :

In [4]:
raw_df = spark.read.option("header", True).csv("avocado.csv")
raw_df.createOrReplaceTempView("CAMPAIGNS")

type(raw_df)

pyspark.sql.dataframe.DataFrame

In [5]:
raw_df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- AveragePrice: string (nullable = true)
 |-- Total Volume: string (nullable = true)
 |-- 4046: string (nullable = true)
 |-- 4225: string (nullable = true)
 |-- 4770: string (nullable = true)
 |-- Total Bags: string (nullable = true)
 |-- Small Bags: string (nullable = true)
 |-- Large Bags: string (nullable = true)
 |-- XLarge Bags: string (nullable = true)
 |-- type: string (nullable = true)
 |-- year: string (nullable = true)
 |-- region: string (nullable = true)



In [7]:
raw_df.toPandas()

22/07/07 15:05:27 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, AveragePrice, Total Volume, 4046, 4225, 4770, Total Bags, Small Bags, Large Bags, XLarge Bags, type, year, region
 Schema: _c0, Date, AveragePrice, Total Volume, 4046, 4225, 4770, Total Bags, Small Bags, Large Bags, XLarge Bags, type, year, region
Expected: _c0 but found: 
CSV file: file:///mnt/c/Users/AhmedBOUGHAMMOURA/LBC/great%20expectations%20demo/avocado.csv


,_c0,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2,2015-12-13,0.93,118220.22,794.7,109149.67,130.5,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,3,2015-12-06,1.08,78992.15,1132.0,71976.41,72.58,5811.16,5677.4,133.76,0.0,conventional,2015,Albany
4,4,2015-11-29,1.28,51039.6,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18244,7,2018-02-04,1.63,17074.83,2046.96,1529.2,0.0,13498.67,13066.82,431.85,0.0,organic,2018,WestTexNewMexico
18245,8,2018-01-28,1.71,13888.04,1191.7,3431.5,0.0,9264.84,8940.04,324.8,0.0,organic,2018,WestTexNewMexico
18246,9,2018-01-21,1.87,13766.76,1191.92,2452.79,727.94,9394.11,9351.8,42.31,0.0,organic,2018,WestTexNewMexico
18247,10,2018-01-14,1.93,16205.22,1527.63,2981.04,727.01,10969.54,10919.54,50.0,0.0,organic,2018,WestTexNewMexico


## Running tests using GE:

In [8]:
#Testing on raw data
from great_expectations.dataset import SparkDFDataset
#Using this command to transform a pyspark imported df to a ge-pyspark-df a crucial problem that's been solved!
raw_test_df = SparkDFDataset(raw_df) 
type(raw_test_df) #Data type

great_expectations.dataset.sparkdf_dataset.SparkDFDataset

## Making sure essential columns exist:

In [13]:
MANDATORY_COLUMNS = [
  "Date",
  "Total Bags",
  "type",
  "region",
    "INTRUDER"
]
for column in MANDATORY_COLUMNS:
    try:
        assert raw_test_df.expect_column_to_exist(column).success, f"Uh oh! Mandatory column {column} does not exist: FAILED"
        print(f"Column {column} exists : PASSED")
    except AssertionError as e:
        print(e)

Column Date exists : PASSED
Column Total Bags exists : PASSED
Column type exists : PASSED
Column region exists : PASSED
Uh oh! Mandatory column INTRUDER does not exist: FAILED


## Testing if the values of a column are within a range: 

In [26]:
test1 = raw_test_df.expect_column_values_to_be_between('AveragePrice', min_value=0.5, max_value=3.0, mostly=0.99)
test1
#test1.result
#test1["success"]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "meta": {},
  "result": {
    "element_count": 18249,
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_count": 11,
    "unexpected_percent": 0.06027727546714888,
    "unexpected_percent_total": 0.06027727546714888,
    "unexpected_percent_nonmissing": 0.06027727546714888,
    "partial_unexpected_list": [
      "0.49",
      "0.46",
      "3.03",
      "3.12",
      "3.25",
      "0.44",
      "0.49",
      "0.48",
      "3.05",
      "3.04",
      "3.17"
    ]
  }
}

## Testing if a column contains unique values: 

In [32]:
test2 = raw_test_df.expect_column_values_to_be_unique("region")
test2["success"]

False

## Running a validation on the ensemble of tests :

In [42]:
raw_test_validation = raw_test_df.validate()
# print( raw_test_df.validate())
print(raw_test_validation.statistics)
print()
print("did the test succeed ? == ",raw_test_validation.success)

{'evaluated_expectations': 9, 'successful_expectations': 5, 'unsuccessful_expectations': 4, 'success_percent': 55.55555555555556}

did the test succeed ? ==  False


22/07/07 16:54:46 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 341964 ms exceeds timeout 120000 ms
22/07/07 16:54:46 WARN SparkContext: Killing executors is not supported by current scheduler.


In [43]:
spark.sparkContext._gateway.shutdown_callback_server()
spark.stop()